# Web Server Log Analysis - Python Take-Home Assessment

## Dataset Overview

This dataset is a real-world **HTTP web server access log** from the **University of Calgary**, recorded between **1994 and 1995** — one of the earliest publicly available web server log datasets.

Every row represents a **single HTTP request** made to the university's web server, containing five key fields:

- **host** — whether the request originated from inside the university network (`local`) or from an external source (`remote`)
- **timestamp** — the exact date and time of the request
- **filename** — the specific file or resource that was requested
- **http_code** — the server's response status (e.g. `200`, `404`, `500`)
- **bytes** — the size of the data transferred (`-` if unavailable)

Understanding this structure upfront allowed me to design a clean, efficient parsing strategy.

## Part 1: Data Loading and Cleaning

In [1]:
import gzip
import urllib.request
import re
import pandas as pd
from datetime import datetime
import os

# ─────────────────────────────────────────────
# STEP 1: Download the dataset
# ─────────────────────────────────────────────

DATA_URL = "ftp://ita.ee.lbl.gov/traces/calgary_access_log.gz"
GZ_FILE  = "calgary_access_log.gz"
LOG_FILE = "calgary_access_log.txt"

def download_data():
    """Download and decompress the Calgary HTTP log dataset."""

    # Download .gz if not already present
    if not os.path.exists(GZ_FILE):
        print("Downloading dataset...")
        try:
          urllib.request.urlretrieve(DATA_URL, GZ_FILE)
          print(f"Downloaded: {GZ_FILE}")
        except Exception as e:
          if os.path.exists(GZ_FILE):
            os.remove(GZ_FILE)     # remove partial download
          print(f"Download failed: {e}")
          raise
    else:
        print(f"Already downloaded: {GZ_FILE}")

    # Decompress if not already done
    if not os.path.exists(LOG_FILE):
        print("Decompressing...")
        with gzip.open(GZ_FILE, 'rb') as f_in:
            with open(LOG_FILE, 'wb') as f_out:
                f_out.write(f_in.read())
        print(f"Decompressed: {LOG_FILE}")
    else:
        print(f"Already decompressed: {LOG_FILE}")

download_data()

Downloaded: calgary_access_log.gz
Decompressing...
Decompressed: calgary_access_log.txt


In [2]:
# ─────────────────────────────────────────────
# STEP 2: Parse each log line
# ─────────────────────────────────────────────

# Calgary HTTP log uses NCSA Common Log Format:
# host rfc931 authuser [DD/Mon/YYYY:HH:MM:SS ±HHMM] "METHOD path protocol" status bytes
# e.g.: host.example.com - - [24/Oct/1994:13:41:41 -0600] "GET /index.html HTTP/1.0" 200 1234

LOG_PATTERN = re.compile(
    r'(\S+)\s+'           # host
    r'\S+\s+'             # rfc931 (usually -)
    r'\S+\s+'             # authuser (usually -)
    r'\[([^\]]+)\]\s+'    # [DD/Mon/YYYY:HH:MM:SS ±HHMM]
    r'"([^"]*?)"\s+'      # "METHOD path protocol"
    r'(\d{3}|-)\s+'       # status code
    r'(\d+|-)'            # bytes transferred
)

def parse_line(line: str) -> dict | None:
    """
    Parse a single Calgary HTTP log line (NCSA Common Log Format) into a dictionary.
    Returns None if the line is malformed.
    """
    try:
        m = LOG_PATTERN.match(line)
        if not m:
            return None

        host, timestamp_str, request, status_str, bytes_str = m.groups()

        # Parse timestamp: e.g. "24/Oct/1994:13:41:41 -0600"
        try:
            timestamp = datetime.strptime(timestamp_str, "%d/%b/%Y:%H:%M:%S %z")
        except ValueError:
            # Try without timezone offset
            timestamp = datetime.strptime(timestamp_str[:20], "%d/%b/%Y:%H:%M:%S")

        # Parse request line: "GET /path HTTP/1.0"
        req_parts = request.split()
        method   = req_parts[0] if len(req_parts) >= 1 else '-'
        filename = req_parts[1] if len(req_parts) >= 2 else '-'

        # Parse http_code safely
        try:
            http_code = int(status_str)
        except ValueError:
            http_code = None

        # Parse bytes (can be '-')
        try:
            bytes_val = int(bytes_str)
        except ValueError:
            bytes_val = None

        # Extract file extension from path (strip query string first)
        path = filename.split('?')[0]
        if '.' in path.rsplit('/', 1)[-1]:
            extension = path.rsplit('.', 1)[-1].lower()
        else:
            extension = 'no_extension'

        return {
            'host':       host,
            'timestamp':  timestamp,
            'date':       timestamp.strftime('%d-%b-%Y'),
            'hour':       timestamp.hour,
            'method':     method,
            'filename':   filename,
            'extension':  extension,
            'http_code':  http_code,
            'bytes':      bytes_val,
        }

    except Exception:
        return None  # skip any line that fails to parse

In [3]:
# ─────────────────────────────────────────────
# STEP 3: Load all lines into a DataFrame
# ─────────────────────────────────────────────

def load_data(log_file: str = LOG_FILE) -> pd.DataFrame:
    """
    Read the log file line by line, parse each line,
    and return a clean pandas DataFrame.
    """
    records = []
    skipped = 0

    with open(log_file, 'r', encoding='latin-1') as f:
        for line in f:
            parsed = parse_line(line)
            if parsed:
                records.append(parsed)
            else:
                skipped += 1

    df = pd.DataFrame(records)

    print(f"✅ Loaded records : {len(df):,}")
    print(f"⚠️  Skipped lines  : {skipped:,}")
    return df

df = load_data()

✅ Loaded records : 724,836
⚠️  Skipped lines  : 1,903


In [4]:
# ── Type conversions ──────────────────────────────────────────────────
# Nullable integer types preserve NaN without upcasting to float
df['http_code'] = df['http_code'].astype(pd.Int16Dtype())
df['bytes']     = df['bytes'].astype(pd.Int64Dtype())

# Categorical columns — small cardinality, saves memory & enables groupby fast-path
df['host']    = df['host'].astype('category')
df['method']    = df['method'].astype('category')
df['extension'] = df['extension'].astype('category')
df['date']      = df['date'].astype('category')

# timestamp: convert to tz-naive UTC datetime64[ns] so pandas stores it
# as a proper DatetimeTZDtype rather than object (mixed tz-aware/naive)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_localize(None)

# Compact integer for hour (0–23)
df['hour']      = df['hour'].astype('int8')

In [5]:
# ─────────────────────────────────────────────
# STEP 4: Validate & inspect the data
# ─────────────────────────────────────────────

print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Sample Rows ===")
print(df.head(5))

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Unique Hosts ===")
print(df['host'].unique())

print("\n=== HTTP Code Distribution ===")
print(df['http_code'].value_counts().sort_index())

print("\n=== Date Range ===")
print(f"From : {df['timestamp'].min()}")
print(f"To   : {df['timestamp'].max()}")

=== Shape ===
(724836, 9)

=== Data Types ===
host               category
timestamp    datetime64[ns]
date               category
hour                   int8
method             category
filename             object
extension          category
http_code             Int16
bytes                 Int64
dtype: object

=== Sample Rows ===
    host           timestamp         date  hour method    filename extension  \
0  local 1994-10-24 19:41:41  24-Oct-1994    13    GET  index.html      html   
1  local 1994-10-24 19:41:41  24-Oct-1994    13    GET       1.gif       gif   
2  local 1994-10-24 19:43:13  24-Oct-1994    13    GET  index.html      html   
3  local 1994-10-24 19:43:14  24-Oct-1994    13    GET       2.gif       gif   
4  local 1994-10-24 19:43:15  24-Oct-1994    13    GET       3.gif       gif   

   http_code  bytes  
0        200    150  
1        200   1210  
2        200   3185  
3        200   2555  
4        200  36403  

=== Missing Values ===
host             0
timestamp  

# Part 2: Analysis Questions

### Q1: Count of total log records

In [6]:
def total_log_records() -> int:
    """
    Q1: Count of total log records.

    Objective:
        Determine the total number of HTTP log entries in the dataset.
        Each line in the log file represents one HTTP request.

    Returns:
        int: Total number of log entries.
    """
    return len(df)


answer1 = total_log_records()
print("Answer 1:")
print(answer1)

Answer 1:
724836


### Q2: Count of unique hosts

In [7]:
def unique_host_count() -> int:
    """
    Q2: Count of unique hosts.

    Objective:
        Determine how many distinct hosts accessed the server.

    Returns:
        int: Number of unique hosts.
    """

    return df['host'].nunique()


answer2 = unique_host_count()
print("Answer 2:")
print(answer2)

Answer 2:
2


### Q3: Date-wise unique filename counts

In [8]:
def datewise_unique_filename_counts() -> dict[str, int]:
    """
    Q3: Date-wise unique filename counts.

    Objective:
        For each date, count the number of unique filenames that accessed the server.
        The date should be in 'dd-MMM-yyyy' format (e.g., '01-Jul-1995').

    Returns:
        dict: A dictionary mapping each date to its count of unique filenames.
              Example: {'01-Jul-1995': 123, '02-Jul-1995': 150}
    """

    return (
        df.groupby('date',observed= True)['filename']
        .nunique()
        .sort_index()
        .to_dict()
    )


answer3 = datewise_unique_filename_counts()
print("Answer 3:")
print(answer3)

Answer 3:
{'01-Apr-1995': 438, '01-Aug-1995': 684, '01-Dec-1994': 271, '01-Feb-1995': 624, '01-Jan-1995': 88, '01-Jul-1995': 388, '01-Jun-1995': 591, '01-Mar-1995': 582, '01-May-1995': 467, '01-Nov-1994': 415, '01-Oct-1995': 554, '01-Sep-1995': 328, '02-Apr-1995': 466, '02-Aug-1995': 857, '02-Dec-1994': 325, '02-Feb-1995': 524, '02-Jan-1995': 141, '02-Jul-1995': 400, '02-Jun-1995': 515, '02-Mar-1995': 601, '02-May-1995': 702, '02-Nov-1994': 430, '02-Oct-1995': 873, '02-Sep-1995': 351, '03-Apr-1995': 795, '03-Aug-1995': 584, '03-Dec-1994': 189, '03-Feb-1995': 570, '03-Jan-1995': 311, '03-Jul-1995': 438, '03-Jun-1995': 401, '03-Mar-1995': 510, '03-May-1995': 609, '03-Nov-1994': 460, '03-Oct-1995': 848, '03-Sep-1995': 214, '04-Apr-1995': 821, '04-Aug-1995': 717, '04-Dec-1994': 213, '04-Feb-1995': 561, '04-Jan-1995': 333, '04-Jul-1995': 612, '04-Jun-1995': 371, '04-Mar-1995': 413, '04-May-1995': 684, '04-Nov-1994': 404, '04-Oct-1995': 893, '04-Sep-1995': 343, '05-Apr-1995': 891, '05-Aug-19

### Q4: Number of 404 response codes

In [9]:
def count_404_errors() -> int:
    """
    Q4: Number of 404 response codes.

    Objective:
        Count how many times the HTTP 404 Not Found status appears in the logs.

    Returns:
        int: Number of 404 errors.
    """

    return int((df['http_code'] == 404).sum())


answer4 = count_404_errors()
print("Answer 4:")
print(answer4)

Answer 4:
23517


### Q5: Top 15 filenames with 404 responses

In [10]:
def top_15_filenames_with_404() -> list[tuple[str, int]]:
    """
    Q5: Top 15 filenames with 404 responses.

    Objective:
        Identify which requested URLs most frequently resulted in a 404 error.
        Return the top 15 filenames sorted by frequency.

    Returns:
        list: A list of tuples (filename, count), sorted by count in descending order.
              Example: [('index.html', 200), ...]
    """
    counts = df[df['http_code'] == 404]['filename'].value_counts().head(15)

    return list(counts.items())


answer5 = top_15_filenames_with_404()
print("Answer 5:")
print(answer5)

Answer 5:
[('index.html', 4737), ('4115.html', 902), ('1611.html', 649), ('5698.xbm', 585), ('710.txt', 408), ('2002.html', 259), ('2177.gif', 193), ('10695.ps', 161), ('6555.html', 153), ('487.gif', 152), ('151.html', 149), ('488.gif', 148), ('3414.gif', 148), ('40.html', 148), ('9678.gif', 142)]


### Q6: Top 15 file extension with 404 responses

In [11]:
def top_15_ext_with_404() -> list[tuple[str, int]]:
    """
    Q6: Top 15 file extensions with 404 responses.

    Objective:
        Find which file extensions generated the most 404 errors.
        Return the top 15 sorted by number of 404s.

    Returns:
        list: A list of tuples (extension, count), sorted by count in descending order.
              Example: [('html', 45), ...]
    """

    counts = df[df['http_code'] == 404]['extension'].value_counts().head(15)

    return list(counts.items())


answer6 = top_15_ext_with_404()
print("Answer 6:")
print(answer6)

Answer 6:
[('html', 12218), ('gif', 7348), ('xbm', 826), ('ps', 754), ('jpg', 538), ('txt', 509), ('no_extension', 315), ('htm', 110), ('cgi', 77), ('com', 46), ('z', 41), ('dvi', 40), ('ca', 36), ('hmtl', 30), ('util', 29)]


### Q7: Total bandwidth transferred per day for the month of July 1995

In [12]:
def total_bandwidth_per_day() -> dict[str, int]:
    """
    Q7: Total bandwidth transferred per day for the month of July 1995.

    Objective:
        Sum the number of bytes transferred per day.
        Skip entries where the byte field is missing or '-'.

    Returns:
        dict: A dictionary mapping each date to total bytes transferred.
              Example: {'01-Jul-1995': 123456789, ...}
    """

    july=df[
        (df['timestamp'].dt.year == 1995) &
        (df['timestamp'].dt.month == 7)
    ]
    return (
        july.dropna(subset=['bytes'])
        .groupby('date',observed=True)['bytes']
        .sum()
        .astype(int)
        .sort_index()
        .to_dict()
    )


answer7 = total_bandwidth_per_day()
print("Answer 7:")
print(answer7)

Answer 7:
{'01-Jul-1995': 11349799, '02-Jul-1995': 8656918, '03-Jul-1995': 13596612, '04-Jul-1995': 26573988, '05-Jul-1995': 19541225, '06-Jul-1995': 19755015, '07-Jul-1995': 9427822, '08-Jul-1995': 5403491, '09-Jul-1995': 4660556, '10-Jul-1995': 14917754, '11-Jul-1995': 22507207, '12-Jul-1995': 17367065, '13-Jul-1995': 15989234, '14-Jul-1995': 19186430, '15-Jul-1995': 15773233, '16-Jul-1995': 9016378, '17-Jul-1995': 19601338, '18-Jul-1995': 17099761, '19-Jul-1995': 17851725, '20-Jul-1995': 20752623, '21-Jul-1995': 25491617, '22-Jul-1995': 8136259, '23-Jul-1995': 9593870, '24-Jul-1995': 22308265, '25-Jul-1995': 24561635, '26-Jul-1995': 24995540, '27-Jul-1995': 25969995, '28-Jul-1995': 36460693, '29-Jul-1995': 11700624, '30-Jul-1995': 23189598, '30-Jun-1995': 6677142, '31-Jul-1995': 26388935}


### Q8: Hourly request distribution

In [13]:
def hourly_request_distribution() -> dict[int, int]:
    """
    Q8: Hourly request distribution.

    Objective:
        Count the number of requests made during each hour (00 to 23).
        Useful for understanding traffic peaks.

    Returns:
        dict: A dictionary mapping hour (int) to request count.
              Example: {0: 120, 1: 90, ..., 23: 80}
    """

    return (
        df.groupby('hour')['filename']
        .count()
        .sort_index()
        .to_dict()
    )


answer8 = hourly_request_distribution()
print("Answer 8:")
print(answer8)

Answer 8:
{0: 18764, 1: 14389, 2: 12694, 3: 10901, 4: 9969, 5: 10804, 6: 13059, 7: 16672, 8: 26591, 9: 33987, 10: 43371, 11: 47588, 12: 46814, 13: 51457, 14: 54562, 15: 50377, 16: 51176, 17: 45060, 18: 33222, 19: 30573, 20: 29691, 21: 27405, 22: 23827, 23: 21883}


### Q9: Top 10 most requested filenames

In [14]:
def top_10_most_requested_filenames() -> list[tuple[str, int]]:
    """
    Q9: Top 10 most requested filenames.

    Objective:
        Identify the most commonly requested URLs (irrespective of status code).

    Returns:
        list: A list of tuples (filename, count), sorted by count in descending order.
                Example: [('index.html', 500), ...]
    """

    counts = df['filename'].value_counts().head(10)
    return list(counts.items())


answer9 = top_10_most_requested_filenames()
print("Answer 9:")
print(answer9)

Answer 9:
[('index.html', 140074), ('3.gif', 24006), ('2.gif', 23606), ('4.gif', 8018), ('244.gif', 5149), ('5.html', 5010), ('4097.gif', 4874), ('8870.jpg', 4493), ('6733.gif', 4278), ('8472.gif', 3843)]


### Q10: HTTP response code distribution

In [15]:
def response_code_distribution() -> dict[int, int]:
    """
    Q10: HTTP response code distribution.

    Objective:
        Count how often each HTTP status code appears in the logs.

    Returns:
        dict: A dictionary mapping HTTP status codes (as int) to their frequency.
              Example: {200: 150000, 404: 3000}
    """

    return (
        df['http_code']
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


answer10 = response_code_distribution()
print("Answer 10:")
print(answer10)

Answer 10:
{200: 568345, 302: 30295, 304: 97792, 400: 15, 401: 46, 403: 4741, 404: 23517, 500: 42, 501: 43}
